# Latent Diffusion with AB-UPT: analysis & uncertainty quantification

Assumes an AB-UPT autoencoder has already been trained (e.g. via `train_autoencoder.py` on SLURM). AE architecture is loaded from the `hp_resolved.yaml` saved next to the checkpoint — no need to re-specify params here.

Flow:
1. load pretrained AB-UPT autoencoder from a checkpoint path
2. extract latents (or reuse cached ones)
3. visualize AE reconstructions on a few test samples
4. train a small DiT denoiser in latent space (fast)
5. sample from the diffusion model and visualize per-point UQ
6. UQ on integrated scalar quantities (mean/peak Cp and |WSS|)

In [1]:
## Setup

# interactive slurm
# salloc --cpus-per-task=28 --mem=50GB --gpus-per-node=1 --time 1-0 srun --pty zsh

# start notebook (token "ggall", no password — same URL every time)
# cd ~/exp/noether && source .venv/bin/activate
# jupyter notebook --no-browser --port=8888 --ip=0.0.0.0 --ServerApp.token=ggall --ServerApp.password=''

In [ ]:
import sys
sys.path.insert(0, "/home/ggalletti/exp/noether")
sys.path.insert(0, "/home/ggalletti/exp/noether/diffusion_uq")

from pathlib import Path

import numpy as np
import torch
import yaml

from noether.training.runners import HydraRunner
from steady_diffusion.autoencoder_experiments import build_abupt_ae_pretrain_config
from steady_diffusion.diffusion import FlowMatchingSchedule, FlowMatchingConfig
from steady_diffusion.latent_diffusion_experiments import (
    build_latent_diffusion_config,
    load_ae_checkpoint,
)
from steady_diffusion.scripts.extract_latents import extract_latents
from steady_diffusion.viz import (
    load_latest_checkpoint,
    plot_field_recon_stl,
    plot_field_uq_stl,
)

print(f"pytorch: {torch.__version__}")
print(f"cuda: {torch.cuda.is_available()} ({torch.cuda.device_count()} device(s))")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB)")


## Config

Point `AE_CHECKPOINT` at the `.th` file.
Architecture params are parsed from the `hp_resolved.yaml` sitting next to the run
directory (two levels up from the checkpoint).


In [3]:
DEVICE = "cuda"
DATASET_ROOT = "/nfs-gpu/research/datasets/drivaerml/preprocessed/subsampled_10x"
STL_ROOT = "/nfs-gpu/research/datasets/drivaerml/raw_surface_data"

# AE_CHECKPOINT = "/home/ggalletti/exp/noether/outputs/abupt_ae_bottleneck/18410_2026-04-15_4s62k/checkpoints/abupt_autoencoder_cp=best_model.loss.test.total_model.th"
# AE_CHECKPOINT = "/home/ggalletti/exp/noether/outputs/abupt_ae_bottleneck/18442_2026-04-15_po8o4/checkpoints/abupt_autoencoder_cp=best_model.loss.test.total_model.th"
# AE_CHECKPOINT = "/home/ggalletti/exp/noether/outputs/abupt_ae_bottleneck/18641_2026-04-16_57nyd/checkpoints/abupt_autoencoder_cp=best_model.loss.test.total_model.th"
AE_CHECKPOINT = "/home/ggalletti/exp/noether/outputs/abupt_ae_bottleneck/18873_2026-04-18_ada5a/checkpoints/abupt_autoencoder_cp=best_model.loss.test.total_model.th"

AE_TAG = Path(AE_CHECKPOINT).parent.parent.name.split("_")[-1]
LATENT_ROOT = f"/home/ggalletti/exp/noether/data/latents_abupt_{AE_TAG}"
DIFF_OUTPUT = f"/home/ggalletti/exp/noether/outputs/latent_diffusion_fm_abupt_{AE_TAG}"
print(f"AE_TAG={AE_TAG}")

## Load pretrained autoencoder

Parse `hp_resolved.yaml` next to the checkpoint and rebuild the config via
`build_abupt_ae_pretrain_config` so architecture matches training exactly.


In [4]:
ckpt_path = Path(AE_CHECKPOINT)
run_dir = ckpt_path.parent.parent  # .../outputs/<ae_kind>_ae/<run_id>/
hp_path = run_dir / "hp_resolved.yaml"
assert hp_path.exists(), f"hp_resolved.yaml not found at {hp_path}"

with open(hp_path) as f:
    hp = yaml.full_load(f)

m = hp["model"]
pl = hp["datasets"]["train"]["pipeline"]

ae_config = build_abupt_ae_pretrain_config(
        dataset_root=DATASET_ROOT,
        output_path=str(run_dir.parent),
        hidden_dim=m["hidden_dim"],
        latent_dim=m["latent_dim"],
        num_heads=m["transformer_block_config"]["num_heads"],
        geometry_depth=m["geometry_depth"],
        num_surface_blocks=m["num_surface_blocks"],
        num_volume_blocks=m["num_volume_blocks"],
        surface_field_dim=m["surface_field_dim"],
        volume_field_dim=m["volume_field_dim"],
        num_geometry_supernodes=pl["num_geometry_supernodes"],
        num_geometry_points=pl["num_geometry_points"],
        num_surface_anchor_points=pl["num_surface_anchor_points"],
        num_volume_anchor_points=pl["num_volume_anchor_points"],
        supernode_radius=m["supernode_pooling_config"].get("radius", 0.25),
        query_ratio=m.get("query_ratio", 0.0),
        latent_num_surface_tokens=m.get("latent_num_surface_tokens"),
        latent_num_volume_tokens=m.get("latent_num_volume_tokens"),
        bottleneck_num_heads=m.get("bottleneck_num_heads", 4),
        bottleneck_mode=m.get("bottleneck_mode"),
        max_epochs=1, batch_size=1,
    )

ae_trainer, ae_model, _, _ = HydraRunner.setup_experiment(device=DEVICE, config=ae_config)
load_ae_checkpoint(ae_model, str(ckpt_path), device=DEVICE)
ae_model.eval().to(DEVICE)

n_params = sum(p.numel() for p in ae_model.parameters())

# token counts + latent compression
n_surf = pl["num_surface_anchor_points"]
n_vol = pl["num_volume_anchor_points"]
latent_dim = m["latent_dim"]
sf_dim, vf_dim = m["surface_field_dim"], m["volume_field_dim"]

K_s = m.get("latent_num_surface_tokens")
K_v = m.get("latent_num_volume_tokens")
bmode = m.get("bottleneck_mode")
has_bottleneck = K_s is not None and K_v is not None
n_latent_tokens = (K_s + K_v) if has_bottleneck else (n_surf + n_vol)

input_values = n_surf * sf_dim + n_vol * vf_dim
latent_values = n_latent_tokens * latent_dim
compression = input_values / latent_values

print(f"Autoencoder loaded — {n_params:,} params")
print(f"hidden_dim={m['hidden_dim']}, latent_dim={latent_dim}, bottleneck_mode={bmode}")
print(f"encode anchors: {n_surf:,} surface + {n_vol:,} volume")
if has_bottleneck:
    print(f"latent tokens ({bmode} bottleneck): {K_s:,} surface + {K_v:,} volume = {n_latent_tokens:,}")
    print(f"  → resolution-independent: latent shape fixed; decode at any positions")
else:
    print(f"latent tokens (1:1 with anchors): {n_latent_tokens:,}")
print(f"input fields (anchor-budget): {input_values:,} values ({n_surf}x{sf_dim} + {n_vol}x{vf_dim})")
print(f"latent: {n_latent_tokens:,}x{latent_dim} = {latent_values:,} values")
print(f"compression vs anchor-budget: {compression:.2f}x ({input_values:,} → {latent_values:,})")

## Extract latents

Skipped if `train_stats.pt` already exists under `LATENT_ROOT`.

In [5]:
if (Path(LATENT_ROOT) / "train_stats.pt").exists():
    print(f"latents already extracted at {LATENT_ROOT}")
else:
    extract_latents(
        ae_config,
        output_root=LATENT_ROOT,
        device=DEVICE,
        batch_size=8,
        checkpoint_path=str(ckpt_path),
    )

stats = torch.load(f"{LATENT_ROOT}/train_stats.pt", weights_only=True)
latent_scale = float(stats["latent_scale"])
latent_mean = stats["latent_mean"]   # (n_tokens, latent_dim)
latent_std = stats["latent_std"]     # (n_tokens, latent_dim)
print(f"latent_scale (scalar, legacy): {latent_scale:.4f}")
print(f"per-token std  min/med/max: {latent_std.mean(-1).min():.3f} / {latent_std.mean(-1).median():.3f} / {latent_std.mean(-1).max():.3f}")
print(f"per-channel std  min/med/max: {latent_std.mean(0).min():.3f} / {latent_std.mean(0).median():.3f} / {latent_std.mean(0).max():.3f}")

## Latent space exploration

Sanity checks on precomputed latents:
1. Per-channel and per-token variance distribution — is `latent_scale` reasonable?
2. t-SNE of per-sample mean latents — are samples separable?
3. Decode precomputed latents back through AE (no diffusion) — isolates AE decode quality from diffusion quality.

In [6]:
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

# load all train + test latents
train_files = sorted((Path(LATENT_ROOT) / "train").glob("*.pt"))
test_files = sorted((Path(LATENT_ROOT) / "test").glob("*.pt"))

def load_latents(files):
    lats = []
    for f in files:
        d = torch.load(f, weights_only=True)
        lats.append(d["latents"])  # (n_tokens, latent_dim)
    return torch.stack(lats)  # (N, n_tokens, latent_dim)

train_lats = load_latents(train_files)
test_lats = load_latents(test_files)
all_lats = torch.cat([train_lats, test_lats], dim=0)
print(f"latents: {all_lats.shape} (train={len(train_lats)}, test={len(test_lats)})")

# per-token z-score normalization (what FM actually trains on)
normed = (all_lats - latent_mean) / latent_std.clamp(min=1e-6)
ch_std = normed.std(dim=(0, 1))   # (latent_dim,) — across (samples, tokens)
tok_std = normed.std(dim=(0, 2))  # (n_tokens,)  — across (samples, channels)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(ch_std.numpy(), bins=50)
axes[0].set_xlabel("per-channel std (after per-token norm)")
axes[0].set_title(f"channel std — ideal ~1.0\nmin={ch_std.min():.2f} med={ch_std.median():.2f} max={ch_std.max():.2f}")
axes[0].axvline(1.0, color="r", ls="--", alpha=0.5)

axes[1].hist(tok_std.numpy(), bins=50)
axes[1].set_xlabel("per-token std (after per-token norm)")
axes[1].set_title(f"token std — ideal ~1.0\nmin={tok_std.min():.2f} med={tok_std.median():.2f} max={tok_std.max():.2f}")
axes[1].axvline(1.0, color="r", ls="--", alpha=0.5)

# t-SNE on per-sample mean latent (use normalized latents)
mean_lats = normed.mean(dim=1).numpy()  # (N, latent_dim)
tsne = TSNE(n_components=2, perplexity=min(30, len(mean_lats) - 1), random_state=0)
emb = tsne.fit_transform(mean_lats)
n_train = len(train_lats)
axes[2].scatter(emb[:n_train, 0], emb[:n_train, 1], s=10, alpha=0.6, label="train")
axes[2].scatter(emb[n_train:, 0], emb[n_train:, 1], s=10, alpha=0.6, label="test")
axes[2].legend()
axes[2].set_title("t-SNE of per-sample mean latent (normalized)")

# flag potential issues
if ch_std.max() > 5 or ch_std.min() < 0.01:
    print(f"WARNING: channel std range [{ch_std.min():.3f}, {ch_std.max():.3f}] — "
          "consider per-channel normalization")
if tok_std.max() / tok_std.min() > 10:
    print(f"WARNING: token std ratio {tok_std.max()/tok_std.min():.1f}x — "
          "some token positions have much more variance than others")

## High-resolution eval pipeline

Provides query positions DIFFERENT from the encode-time anchors stored in the latent files. Resolution-independent decoding for the bottleneck AE; query-decoding for the non-bottleneck AE.


In [7]:
# ── High-resolution eval pipeline (50K + 50K positions) ───────────────
# Provides query positions DIFFERENT from the encode-time positions stored in
# the latent files. Demonstrates resolution-independent decoding when the AE
# uses a token bottleneck. Without bottleneck, we still decode at new positions
# via the decoder's `query_*_position` mechanism.

N_EVAL_SURFACE = 200_000
N_EVAL_VOLUME = 200_000

eval_ae_config = build_abupt_ae_pretrain_config(
    dataset_root=DATASET_ROOT,
    output_path="./outputs/_eval_ae_hires_tmp",
    hidden_dim=m["hidden_dim"],
    latent_dim=m["latent_dim"],
    num_heads=m["transformer_block_config"]["num_heads"],
    geometry_depth=m["geometry_depth"],
    num_surface_blocks=m["num_surface_blocks"],
    num_volume_blocks=m["num_volume_blocks"],
    surface_field_dim=m["surface_field_dim"],
    volume_field_dim=m["volume_field_dim"],
    num_geometry_supernodes=pl["num_geometry_supernodes"],
    num_geometry_points=pl["num_geometry_points"],
    num_surface_anchor_points=N_EVAL_SURFACE,
    num_volume_anchor_points=N_EVAL_VOLUME,
    supernode_radius=m["supernode_pooling_config"].get("radius", 0.25),
    query_ratio=0.0,
    latent_num_surface_tokens=m.get("latent_num_surface_tokens"),
    latent_num_volume_tokens=m.get("latent_num_volume_tokens"),
    bottleneck_num_heads=m.get("bottleneck_num_heads", 4),
    bottleneck_mode=m.get("bottleneck_mode"),
    max_epochs=1, batch_size=1,
)
eval_ae_tr, _, _, _ = HydraRunner.setup_experiment(device="cpu", config=eval_ae_config)
ds_test_hires = eval_ae_tr.data_container.get_dataset("test")
print(f"high-res eval pipeline: {N_EVAL_SURFACE} surface + {N_EVAL_VOLUME} volume")


def get_hires_batch(idx: int):
    """Sample idx from the 50K-anchor pipeline. Returns positions + GT fields
    at points DIFFERENT from the ones the latent was encoded at."""
    b = ds_test_hires.pipeline([ds_test_hires[idx]])
    return {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in b.items()}


# helpers used by viz + UQ cells below
def _get_normalizers(ds):
    base = ds
    while hasattr(base, "dataset") and not hasattr(base, "normalizers"):
        base = base.dataset
    return base.normalizers


def _design_id(ds, idx):
    base = ds
    while hasattr(base, "indices") and hasattr(base, "dataset"):
        idx = base.indices[idx]
        base = base.dataset
    while hasattr(base, "dataset") and not hasattr(base, "design_ids"):
        base = base.dataset
    return int(base.design_ids[idx])


field_normalizers = _get_normalizers(ds_test_hires)
print(f"normalizers: {list(field_normalizers.keys())}")


def denormalize(pred: dict, normalizers: dict) -> dict:
    """Apply inverse normalization to bring predictions back to physical units."""
    return {k: (normalizers[k].inverse(v) if k in normalizers else v) for k, v in pred.items()}


def _super_pos_from_ref(ref):
    """pull (super_pos_surf, super_pos_vol) from a saved latent payload, or
    (None, None) if the AE was trained without sampled bottleneck."""
    sp_s = ref.get("super_position_surface")
    sp_v = ref.get("super_position_volume")
    if sp_s is not None:
        sp_s = sp_s.unsqueeze(0).to(DEVICE)
    if sp_v is not None:
        sp_v = sp_v.unsqueeze(0).to(DEVICE)
    return sp_s, sp_v


def decode_at_hires(latents, hires_batch, *, encode_surf=None, encode_vol=None,
                    encode_geom_pos=None, encode_geom_sn_idx=None,
                    super_pos_surf=None, super_pos_vol=None):
    """Decode latent at the high-res positions in `hires_batch`.

    bottleneck case: latent has fixed K tokens. We pass the 50K hires
        positions as `surface_anchor_position` directly — the decoder
        cross-attends position embeddings to the K latent tokens and
        outputs predictions at those 50K points. For sampled mode,
        `super_pos_*` carry the per-token positions and must be supplied.

    non-bottleneck case: latent is 1:1 with encode anchors, so we MUST pass
        the encode-time anchors as `surface_anchor_position` to align the
        latent tokens spatially. The 50K hires positions go through the
        decoder's `query_*_position` mechanism (cross-attn to anchor reps).
    """
    surf_hires = hires_batch["surface_anchor_position"]
    vol_hires = hires_batch["volume_anchor_position"]
    geom_hires = hires_batch["geometry_position"]
    geom_sn_hires = hires_batch["geometry_supernode_idx"]

    if ae_model.use_token_bottleneck:
        pred = ae_model.decode(
            latents, surf_hires, vol_hires, geom_hires, geom_sn_hires,
            super_position_surface=super_pos_surf,
            super_position_volume=super_pos_vol,
        )
        return pred
    else:
        assert encode_surf is not None and encode_vol is not None, \
            "non-bottleneck AE needs encode-time anchor positions to align latent tokens"
        pred = ae_model.decode(
            latents, encode_surf, encode_vol, encode_geom_pos, encode_geom_sn_idx,
            query_surface_position=surf_hires, query_volume_position=vol_hires,
        )
        # query_* is the prediction at hires positions — strip prefix for uniform keys
        return {k.removeprefix("query_"): v for k, v in pred.items() if k.startswith("query_")}

In [ ]:
# Decode precomputed latents (a) at encode-time anchors and (b) at 50K
# DIFFERENT positions from the high-res pipeline. If the bottleneck AE is
# resolution-independent, NMSE at 50K new points should be on par with the
# at-anchor reconstruction (small extra error from interpolation).
#
# Vorticity uses logscale=True normalization (sign*log1p). Physical-space
# NMSE explodes because `inverse` applies exp, amplifying any residual.
# Print NMSE at three stages:
#   norm  - network output space (post mean/std, post log for vorticity)
#   log   - after undoing mean/std only (pre-exp for vorticity)
#   phys  - after full inverse (SI units)

N_RECON = 4
recon_idxs = list(range(min(N_RECON, len(test_files))))


def _nmse(p, t):
    return ((p - t) ** 2).mean().item() / max((t ** 2).mean().item(), 1e-20)


def _nmse_triplet(p_norm, t_norm, fname):
    """(norm, log, phys). log == undo mean/std only. Same as phys for
    non-logscale fields, but kept so the table reads consistently."""
    norm = field_normalizers.get(fname)
    nz = _nmse(p_norm, t_norm)
    if norm is None:
        return nz, float("nan"), float("nan")
    tx = norm.transforms[0] if hasattr(norm, "transforms") else norm
    scale = tx.scale if torch.is_tensor(tx.scale) else torch.tensor(tx.scale, dtype=p_norm.dtype)
    shift = tx.shift if torch.is_tensor(tx.shift) else torch.tensor(tx.shift, dtype=p_norm.dtype)
    p_log = p_norm / scale - shift
    t_log = t_norm / scale - shift
    nl = _nmse(p_log, t_log)
    np_ = _nmse(norm.inverse(p_norm), norm.inverse(t_norm))
    return nz, nl, np_


ae_model.eval()
for idx in recon_idxs:
    ref = torch.load(test_files[idx], weights_only=True)
    z = ref["latents"].unsqueeze(0).to(DEVICE)
    s_anchor = ref["surface_anchor_position"].unsqueeze(0).to(DEVICE)
    v_anchor = ref["volume_anchor_position"].unsqueeze(0).to(DEVICE)
    geom_pos = ref.get("geometry_position", torch.empty(0, 3)).to(DEVICE)
    geom_sn_idx = ref.get("geometry_supernode_idx", torch.empty(0, dtype=torch.long)).to(DEVICE)

    print(f"\n--- test latent [{idx}] ---")

    # (a) at encode-time anchor positions
    sp_s, sp_v = _super_pos_from_ref(ref)
    with torch.no_grad():
        pred_anchor = ae_model.decode(
            z, s_anchor, v_anchor, geom_pos, geom_sn_idx,
            super_position_surface=sp_s, super_position_volume=sp_v,
        )
    gt_s = ref.get("surface_targets", {})
    gt_v = ref.get("volume_targets", {})

    print("  at encode-time anchors:")
    for fname in ["surface_pressure", "surface_friction", "volume_pressure",
                  "volume_velocity", "volume_vorticity"]:
        tgt_dict = gt_s if fname.startswith("surface") else gt_v
        tgt_key = f"{fname}_target"
        if tgt_key in tgt_dict and fname in pred_anchor:
            t_norm = tgt_dict[tgt_key]
            p_norm = pred_anchor[fname][0].cpu()
            nz, nl, np_ = _nmse_triplet(p_norm, t_norm, fname)
            print(f"    {fname:<18s} NMSE norm={nz:.4f}  log={nl:.4f}  phys={np_:.4f}")

    # (b) at 50K NEW positions from high-res pipeline
    hires = get_hires_batch(idx)
    with torch.no_grad():
        pred_hires = decode_at_hires(
            z, hires,
            encode_surf=s_anchor, encode_vol=v_anchor,
            encode_geom_pos=geom_pos, encode_geom_sn_idx=geom_sn_idx,
            super_pos_surf=sp_s, super_pos_vol=sp_v,
        )
    print(f"  at {N_EVAL_SURFACE}/{N_EVAL_VOLUME} NEW positions (resolution-independence test):")
    for fname in ["surface_pressure", "surface_friction", "volume_pressure",
                  "volume_velocity", "volume_vorticity"]:
        tgt_key = f"{fname}_target"
        if tgt_key not in hires or fname not in pred_hires:
            continue
        t_norm = hires[tgt_key][0].cpu()
        p_norm = pred_hires[fname][0].cpu()
        nz, nl, np_ = _nmse_triplet(p_norm, t_norm, fname)
        print(f"    {fname:<18s} NMSE norm={nz:.4f}  log={nl:.4f}  phys={np_:.4f}")


## AE reconstruction quality

Run AE on a few random test samples, plot Cp + |WSS| scatter at surface anchors, then compute per-field MSE on the full test set.

In [9]:
ds_test = ae_trainer.data_container.get_dataset("test")
rng = np.random.default_rng(0)
viz_idxs = rng.choice(len(ds_test), size=2, replace=False).tolist()
print(f"visualizing test indices: {viz_idxs}")

for idx in viz_idxs:
    design_id = _design_id(ds_test, idx)
    stl_path = f"{STL_ROOT}/run_{design_id}/drivaer_{design_id}.stl"

    batch = ds_test.pipeline([ds_test[idx]])
    batch = {k: v.to(DEVICE) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}

    with torch.no_grad():
        pred = ae_model(
            geometry_position=batch["geometry_position"],
            geometry_supernode_idx=batch["geometry_supernode_idx"],
            geometry_batch_idx=batch["geometry_batch_idx"],
            surface_anchor_position=batch["surface_anchor_position"],
            volume_anchor_position=batch["volume_anchor_position"],
            surface_pressure_target=batch.get("surface_pressure_target"),
            surface_friction_target=batch.get("surface_friction_target"),
            volume_pressure_target=batch.get("volume_pressure_target"),
            volume_velocity_target=batch.get("volume_velocity_target"),
            volume_vorticity_target=batch.get("volume_vorticity_target"),
        )

    positions = batch["surface_anchor_position"][0].cpu().numpy()

    cp_t = batch["surface_pressure_target"][0].cpu().numpy().squeeze(-1)
    cp_p = pred["surface_pressure"][0].cpu().numpy().squeeze(-1)
    print(f"\n=== test[{idx}] run_{design_id} — Cp ===")
    plot_field_recon_stl(positions, cp_t, cp_p, stl_path=stl_path,
                         field_name="Cp", index=idx, view="front",
                         align="normalizer", pos_normalizer=field_normalizers["surface_position"])

    wss_t = np.linalg.norm(batch["surface_friction_target"][0].cpu().numpy(), axis=-1)
    wss_p = np.linalg.norm(pred["surface_friction"][0].cpu().numpy(), axis=-1)
    print(f"=== test[{idx}] run_{design_id} — |WSS| ===")
    plot_field_recon_stl(positions, wss_t, wss_p, stl_path=stl_path,
                         field_name="|WSS|", index=idx, view="front",
                         align="normalizer", pos_normalizer=field_normalizers["surface_position"])

## Train latent diffusion (flow matching)

In [ ]:
TRAIN_NEW = True

K_s = m.get("latent_num_surface_tokens")
K_v = m.get("latent_num_volume_tokens")
has_bottleneck = K_s is not None and K_v is not None

if has_bottleneck:
    n_surf_tokens = K_s          # latent count, not anchor count
    use_rope = bmode == "sampled"  # sampled has real 3D positions
else:
    n_surf_tokens = pl.get("num_surface_anchor_points", 1024)
    use_rope = True              # RoPE over anchor positions

config_diff = build_latent_diffusion_config(
    latent_root=LATENT_ROOT,
    ae_config=ae_config,
    output_path=DIFF_OUTPUT,
    paradigm="flow_matching",
    latent_scale=latent_scale,
    latent_dim=m["latent_dim"],
    hidden_dim=384,
    denoiser_depth=4,
    denoiser_heads=6,
    denoiser_use_rope=use_rope,
    num_surface_tokens=n_surf_tokens,
    condition_dim=m["latent_dim"],
    max_epochs=500,
    batch_size=32,
    precision="bfloat16",
    lr=4e-3,
    num_workers=2,
    save_checkpoints=TRAIN_NEW,
    adaln_zero_std=0.001,
)

if TRAIN_NEW:
    HydraRunner.main(device=DEVICE, config=config_diff)
else:
    print(f"skip training — will load latest checkpoint from {DIFF_OUTPUT}")


In [ ]:
_, diff_model, _, _ = HydraRunner.setup_experiment(device=DEVICE, config=config_diff)
if diff_model.autoencoder is not None:
    load_ae_checkpoint(diff_model.autoencoder, str(ckpt_path), device=DEVICE)
load_latest_checkpoint(diff_model.denoiser, DIFF_OUTPUT, "denoiser", DEVICE)
diff_model.eval().to(DEVICE)

schedule = FlowMatchingSchedule(FlowMatchingConfig(minibatch_ot=False)).to(DEVICE)

print(f"denoiser: {sum(p.numel() for p in diff_model.denoiser.parameters()):,} params")

## Per-point UQ on val samples

For each of `N_SHOW` random val geometries, draw `N_UQ` latents from the diffusion prior, decode each through the AB-UPT AE, then plot GT / mean / std / |error|.

Pearson correlation between std (UQ) and |error| is shown in the std panel — high correlation means the diffusion's spread tracks where predictions are unreliable.

In [ ]:
N_UQ = 8
N_SHOW = 2
SAMPLING_STEPS = 10

val_latent_files = sorted((Path(LATENT_ROOT) / "test").glob("*.pt"))
n_avail = min(len(ds_test_hires), len(val_latent_files))
rng = np.random.default_rng(0)
sample_idxs = rng.choice(n_avail, size=min(N_SHOW, n_avail), replace=False).tolist()
print(f"UQ on test indices: {sample_idxs} — decoding at {N_EVAL_SURFACE} NEW surface points")

# per-token denormalization: FM samples in z-scored space → invert to raw latent
_lat_mean = latent_mean.to(DEVICE)
_lat_std = latent_std.clamp(min=1e-6).to(DEVICE)

def denorm_latent(z: torch.Tensor) -> torch.Tensor:
    """Invert per-token z-score: raw = z * std + mean."""
    return z * _lat_std + _lat_mean


def sample_field_ensemble(idx: int):
    """Draw N_UQ diffusion latents, decode each at 50K NEW high-res positions.

    For the bottleneck AE this is true resolution-independent decoding —
    the latent (K tokens) is generated by diffusion, then expanded to
    arbitrary points via the decoder. For the non-bottleneck AE we still
    feed encode-time anchors so latent tokens align spatially, and use the
    decoder's query mechanism for the 50K hires points.
    """
    ref = torch.load(val_latent_files[idx], weights_only=True)
    latent_shape = ref["latents"].shape

    # encode-time positions (needed only for non-bottleneck path)
    s_anchor = ref["surface_anchor_position"].unsqueeze(0).to(DEVICE)
    v_anchor = ref["volume_anchor_position"].unsqueeze(0).to(DEVICE)
    geom_pos_enc = ref["geometry_position"].to(DEVICE)
    geom_sn_idx_enc = ref["geometry_supernode_idx"].to(DEVICE)

    # high-res eval positions + GT (DIFFERENT from encode-time)
    hires = get_hires_batch(idx)

    # denoiser conditioning positions: must match the LATENT token count,
    # so use the saved supernode_positions (K tokens for bottleneck AE,
    # N_anchor tokens for non-bottleneck), NOT a fresh anchor concat.
    cond_pos = ref["supernode_positions"].unsqueeze(0).to(DEVICE)

    # super-token positions for grid/sampled bottleneck decode (None otherwise)
    sp_s, sp_v = _super_pos_from_ref(ref)

    samples: dict[str, list[np.ndarray]] = {}
    with torch.no_grad():
        for _ in range(N_UQ):
            model_fn = lambda x, t, c, sp=cond_pos: diff_model(
                x, timestep=t, supernode_positions=sp,
            )
            z = schedule.sample((1, *latent_shape), model_fn, steps=SAMPLING_STEPS)
            z_raw = denorm_latent(z)
            pred = decode_at_hires(
                z_raw, hires,
                encode_surf=s_anchor, encode_vol=v_anchor,
                encode_geom_pos=geom_pos_enc, encode_geom_sn_idx=geom_sn_idx_enc,
                super_pos_surf=sp_s, super_pos_vol=sp_v,
            )
            pred_phys = denormalize({k: v[0].cpu() for k, v in pred.items()}, field_normalizers)
            for fname, arr in pred_phys.items():
                samples.setdefault(fname, []).append(arr.numpy())

    return ref, hires, {k: np.stack(v) for k, v in samples.items()}


for idx in sample_idxs:
    design_id = _design_id(ds_test_hires, idx)
    stl_path = f"{STL_ROOT}/run_{design_id}/drivaer_{design_id}.stl"
    print(f"\n=== test[{idx}] run_{design_id} ===")
    ref, hires, samples = sample_field_ensemble(idx)
    positions = hires["surface_anchor_position"][0].cpu().numpy()

    sp_target_phys = field_normalizers["surface_pressure"].inverse(
        hires["surface_pressure_target"][0].cpu()
    ).numpy().squeeze(-1)
    sp = samples["surface_pressure"].squeeze(-1)
    plot_field_uq_stl(positions, sp_target_phys, sp.mean(0), sp.std(0),
                      stl_path=stl_path, field_name="Cp", index=idx, view="front",
                      align="normalizer", pos_normalizer=field_normalizers["surface_position"])

    sf_target_phys = field_normalizers["surface_friction"].inverse(
        hires["surface_friction_target"][0].cpu()
    ).numpy()
    sf_target_mag = np.linalg.norm(sf_target_phys, axis=-1)
    sf_mag = np.linalg.norm(samples["surface_friction"], axis=-1)
    plot_field_uq_stl(positions, sf_target_mag, sf_mag.mean(0), sf_mag.std(0),
                      stl_path=stl_path, field_name="|WSS|", index=idx, view="front",
                      align="normalizer", pos_normalizer=field_normalizers["surface_position"])

## Full-mesh evaluation (latent diffusion)

Mirrors `evaluate_legacy_ddpm_full_mesh.py` for the latent pipeline:
sample latent → per-token denorm → chunk-decode at **entire raw mesh** → metrics in physical space.

Loads raw `.pt` field tensors (physical units) from the preprocessed dataset and decodes
the full mesh via query-position chunking (no subsampling). Reports single-draw and
mean-of-draws rel-L2.

In [ ]:
import time
import math

FULL_MESH_FIELDS = {
    "surface_position": "surface_position_vtp.pt",
    "surface_pressure": "surface_pressure.pt",
    "surface_friction": "surface_wallshearstress.pt",
    "volume_position": "volume_cell_position.pt",
    "volume_pressure": "volume_cell_totalpcoeff.pt",
    "volume_velocity": "volume_cell_velocity.pt",
    "volume_vorticity": "volume_cell_vorticity.pt",
}

# full mesh: no subsampling — chunk the decoder query over CHUNK points
CHUNK = 200_000
N_DRAWS_EVAL = 3
N_STEPS_EVAL = 10
N_EVAL_GEOMS = 5

pos_norm_s = field_normalizers["surface_position"]
pos_norm_v = field_normalizers["volume_position"]


def load_full_mesh(sample_dir):
    out = {}
    for k, fname in FULL_MESH_FIELDS.items():
        t = torch.load(Path(sample_dir) / fname, weights_only=False, map_location="cpu")
        if t.dim() == 1:
            t = t.unsqueeze(-1)
        out[k] = t.float()
    return out


def _relL2(p, t):
    return (((p - t) ** 2).sum().sqrt() / (t ** 2).sum().sqrt().clamp(min=1e-12)).item()


def _relL2_triplet(p_norm, t_norm, fname):
    """(norm, log, phys) rel-L2. log = undo mean/std only (pre-exp for logscale
    fields). For non-logscale fields, log == phys up to affine — kept for
    table consistency."""
    norm = field_normalizers.get(fname)
    nz = _relL2(p_norm, t_norm)
    if norm is None:
        return nz, float("nan"), float("nan")
    tx = norm.transforms[0] if hasattr(norm, "transforms") else norm
    scale = tx.scale if torch.is_tensor(tx.scale) else torch.tensor(tx.scale, dtype=p_norm.dtype)
    shift = tx.shift if torch.is_tensor(tx.shift) else torch.tensor(tx.shift, dtype=p_norm.dtype)
    p_log = p_norm / scale - shift
    t_log = t_norm / scale - shift
    nl = _relL2(p_log, t_log)
    np_ = _relL2(norm.inverse(p_norm), norm.inverse(t_norm))
    return nz, nl, np_


def chunked_decode(z_raw, surf_full, vol_full, geom_pos, geom_sn_idx,
                   s_anchor, v_anchor, geom_pos_enc, geom_sn_idx_enc,
                   sp_s, sp_v):
    """Decode z_raw at full-mesh query positions in CHUNK-sized slices.

    surf_full / vol_full: (1, N, 3) query positions (already normalized, on DEVICE).
    Runs surface+volume jointly per chunk; when one side is exhausted, passes
    a 1-point dummy slice (last position) and discards its outputs.
    Returns dict field -> (N_field, dim_f) CPU tensors."""
    N_s = surf_full.shape[1]
    N_v = vol_full.shape[1]
    n_chunks = max(math.ceil(N_s / CHUNK), math.ceil(N_v / CHUNK))
    out_chunks: dict[str, list[torch.Tensor]] = {}
    for c in range(n_chunks):
        ss, se = c * CHUNK, min((c + 1) * CHUNK, N_s)
        vs, ve = c * CHUNK, min((c + 1) * CHUNK, N_v)
        has_s, has_v = ss < N_s, vs < N_v
        s_slice = surf_full[:, ss:se] if has_s else surf_full[:, -1:]
        v_slice = vol_full[:, vs:ve] if has_v else vol_full[:, -1:]

        eval_b = {
            "surface_anchor_position": s_slice,
            "volume_anchor_position": v_slice,
            "geometry_position": geom_pos,
            "geometry_supernode_idx": geom_sn_idx,
        }
        pred = decode_at_hires(
            z_raw, eval_b,
            encode_surf=s_anchor, encode_vol=v_anchor,
            encode_geom_pos=geom_pos_enc, encode_geom_sn_idx=geom_sn_idx_enc,
            super_pos_surf=sp_s, super_pos_vol=sp_v,
        )
        for f, t in pred.items():
            if f.startswith("surface"):
                if not has_s:
                    continue
                out_chunks.setdefault(f, []).append(t[0, :se - ss].cpu())
            elif f.startswith("volume"):
                if not has_v:
                    continue
                out_chunks.setdefault(f, []).append(t[0, :ve - vs].cpu())
    return {f: torch.cat(lst, 0) for f, lst in out_chunks.items()}


ALL_FIELDS = ["surface_pressure", "surface_friction",
              "volume_pressure", "volume_velocity", "volume_vorticity"]
SPACES = ["norm", "log", "phys"]
single_draw = {f: {s: [] for s in SPACES} for f in ALL_FIELDS}
mean_draw   = {f: {s: [] for s in SPACES} for f in ALL_FIELDS}

eval_idxs = list(range(min(N_EVAL_GEOMS, len(val_latent_files))))

for idx in eval_idxs:
    design_id = _design_id(ds_test_hires, idx)
    sample_dir = f"{DATASET_ROOT}/run_{design_id}"

    fm = load_full_mesh(sample_dir)
    N_s, N_v = fm["surface_position"].shape[0], fm["volume_position"].shape[0]

    # full mesh, no subsampling
    surf_pos = pos_norm_s(fm["surface_position"]).unsqueeze(0).to(DEVICE)
    vol_pos  = pos_norm_v(fm["volume_position"]).unsqueeze(0).to(DEVICE)

    gt_phys = {
        "surface_pressure": fm["surface_pressure"],
        "surface_friction": fm["surface_friction"],
        "volume_pressure":  fm["volume_pressure"],
        "volume_velocity":  fm["volume_velocity"],
        "volume_vorticity": fm["volume_vorticity"],
    }
    gt_norm = {f: field_normalizers[f](gt_phys[f]) for f in ALL_FIELDS}

    ref = torch.load(val_latent_files[idx], weights_only=True)
    latent_shape = ref["latents"].shape
    cond_pos = ref["supernode_positions"].unsqueeze(0).to(DEVICE)
    sp_s, sp_v = _super_pos_from_ref(ref)
    s_anchor = ref["surface_anchor_position"].unsqueeze(0).to(DEVICE)
    v_anchor = ref["volume_anchor_position"].unsqueeze(0).to(DEVICE)
    geom_pos_enc = ref["geometry_position"].to(DEVICE)
    geom_sn_idx_enc = ref["geometry_supernode_idx"].to(DEVICE)

    hires_batch = get_hires_batch(idx)
    geom_pos = hires_batch["geometry_position"]
    geom_sn_idx = hires_batch["geometry_supernode_idx"]

    # running sum of per-draw predictions for mean-of-draws metric
    sum_pred_norm: dict[str, torch.Tensor] = {}
    print(f"\n=== test[{idx}] run_{design_id} (full mesh: {N_s:,}S + {N_v:,}V, "
          f"{math.ceil(max(N_s,N_v)/CHUNK)} chunks) ===")

    t0 = time.time()
    for d in range(N_DRAWS_EVAL):
        with torch.no_grad():
            model_fn = lambda x, t, c, sp=cond_pos: diff_model(
                x, timestep=t, supernode_positions=sp,
            )
            z = schedule.sample((1, *latent_shape), model_fn, steps=N_STEPS_EVAL)
            z_raw = denorm_latent(z)
            pred = chunked_decode(
                z_raw, surf_pos, vol_pos, geom_pos, geom_sn_idx,
                s_anchor, v_anchor, geom_pos_enc, geom_sn_idx_enc,
                sp_s, sp_v,
            )

        for f in ALL_FIELDS:
            if f not in pred:
                continue
            p_n = pred[f]
            t_n = gt_norm[f]
            nz, nl, np_ = _relL2_triplet(p_n, t_n, f)
            single_draw[f]["norm"].append(nz)
            single_draw[f]["log"].append(nl)
            single_draw[f]["phys"].append(np_)
            if f in sum_pred_norm:
                sum_pred_norm[f] += p_n
            else:
                sum_pred_norm[f] = p_n.clone()

        parts = []
        for f in ["surface_pressure", "volume_pressure", "volume_vorticity"]:
            sd = single_draw[f]
            if sd["phys"]:
                parts.append(f"{f}={sd['phys'][-1]*100:.1f}")
        print(f"  draw {d+1}/{N_DRAWS_EVAL} ({time.time()-t0:.0f}s) phys%: " + ", ".join(parts))

    for f in ALL_FIELDS:
        if f not in sum_pred_norm:
            continue
        mean_pred_norm = sum_pred_norm[f] / N_DRAWS_EVAL
        nz, nl, np_ = _relL2_triplet(mean_pred_norm, gt_norm[f], f)
        mean_draw[f]["norm"].append(nz)
        mean_draw[f]["log"].append(nl)
        mean_draw[f]["phys"].append(np_)

    parts = []
    for f in ["surface_pressure", "volume_pressure", "volume_vorticity"]:
        md = mean_draw[f]
        if md["phys"]:
            parts.append(f"{f}={md['phys'][-1]*100:.1f}")
    print(f"  mean-of-{N_DRAWS_EVAL} phys%: " + ", ".join(parts))

    # free GPU query tensors before next geom
    del surf_pos, vol_pos, sum_pred_norm

# ── summary ────────────────────────────────────────────────────────────
bar = "=" * 80
print(f"\n{bar}")
print(f"LATENT DIFFUSION FULL-MESH EVAL ({len(eval_idxs)} geoms, "
      f"{N_DRAWS_EVAL} draws, {N_STEPS_EVAL} steps, "
      f"entire mesh, chunk={CHUNK//1000}K) — rel-L2 %")
print(bar)
hdr = f"    {'field':>25s}  {'norm':>8s}  {'log':>8s}  {'phys':>8s}"
for label, m in [("Single-draw", single_draw), (f"Mean-of-{N_DRAWS_EVAL}", mean_draw)]:
    print(f"\n  {label}:")
    print(hdr)
    for f in ALL_FIELDS:
        if not m[f]["phys"]:
            continue
        nz = np.mean(m[f]["norm"]) * 100
        nl = np.mean(m[f]["log"]) * 100
        np_ = np.mean(m[f]["phys"]) * 100
        print(f"    {f:>25s}  {nz:>7.2f}%  {nl:>7.2f}%  {np_:>7.2f}%")


## UQ on integrated quantities

Per-point UQ is noisy; integrated scalars are what a CFD user actually cares about (drag, lift, peak loads). Since surface normals aren't in the pipeline, we use **scalar summaries** as drag/lift proxies:

- `mean(Cp)` — pressure integral proxy
- `max(Cp)` / `min(Cp)` — stagnation / suction peak
- `mean(|WSS|)` — friction integral proxy
- `max(|WSS|)` — peak shear

For each of `N_CAL` val geometries, draw `N_UQ_INT` diffusion samples, compute the scalars, and plot **target vs. predicted-mean ± predicted-std** across the validation set. We report:

- **R²** of predicted mean vs target
- **coverage @ 1σ** (fraction of targets within mean ± 1·std) — ideal ≈ 0.68
- **mean |z-score|** `|target − mean| / std` — ideal ≈ 0.8 (half-normal expectation)

In [ ]:
import matplotlib.pyplot as plt

N_CAL = 20     # test geometries to evaluate
N_UQ_INT = 8   # diffusion samples per geometry

n_cal = min(N_CAL, len(val_latent_files), len(ds_test_hires))
rng = np.random.default_rng(1)
cal_idxs = rng.choice(n_cal, size=n_cal, replace=False).tolist()
print(f"integrated-quantity UQ on {n_cal} test geometries (decoding at {N_EVAL_SURFACE} NEW surface points)")

SCALARS = {
    "mean Cp":    lambda cp, wss: cp.mean(),
    "max Cp":     lambda cp, wss: cp.max(),
    "min Cp":     lambda cp, wss: cp.min(),
    "mean |WSS|": lambda cp, wss: wss.mean(),
    "max |WSS|":  lambda cp, wss: wss.max(),
}

targets: dict[str, list[float]] = {k: [] for k in SCALARS}
pred_means: dict[str, list[float]] = {k: [] for k in SCALARS}
pred_stds: dict[str, list[float]] = {k: [] for k in SCALARS}

for j, idx in enumerate(cal_idxs):
    ref = torch.load(val_latent_files[idx], weights_only=True)
    latent_shape = ref["latents"].shape

    s_anchor = ref["surface_anchor_position"].unsqueeze(0).to(DEVICE)
    v_anchor = ref["volume_anchor_position"].unsqueeze(0).to(DEVICE)
    geom_pos_enc = ref["geometry_position"].to(DEVICE)
    geom_sn_idx_enc = ref["geometry_supernode_idx"].to(DEVICE)
    cond_pos = ref["supernode_positions"].unsqueeze(0).to(DEVICE)
    sp_s, sp_v = _super_pos_from_ref(ref)

    hires = get_hires_batch(idx)

    # GT scalars at the 50K NEW positions, in physical units
    cp_t_phys = field_normalizers["surface_pressure"].inverse(
        hires["surface_pressure_target"][0].cpu()
    ).numpy().squeeze(-1)
    sf_t_phys = field_normalizers["surface_friction"].inverse(
        hires["surface_friction_target"][0].cpu()
    ).numpy()
    wss_t = np.linalg.norm(sf_t_phys, axis=-1)
    for name, fn in SCALARS.items():
        targets[name].append(float(fn(cp_t_phys, wss_t)))

    per_sample: dict[str, list[float]] = {name: [] for name in SCALARS}
    with torch.no_grad():
        for _ in range(N_UQ_INT):
            model_fn = lambda x, t, c, sp=cond_pos: diff_model(
                x, timestep=t, supernode_positions=sp,
            )
            z = schedule.sample((1, *latent_shape), model_fn, steps=SAMPLING_STEPS)
            z_raw = denorm_latent(z)
            pred = decode_at_hires(
                z_raw, hires,
                encode_surf=s_anchor, encode_vol=v_anchor,
                encode_geom_pos=geom_pos_enc, encode_geom_sn_idx=geom_sn_idx_enc,
                super_pos_surf=sp_s, super_pos_vol=sp_v,
            )
            pred_phys = denormalize({k: v[0].cpu() for k, v in pred.items()}, field_normalizers)
            cp = pred_phys["surface_pressure"].numpy().squeeze(-1)
            wss = np.linalg.norm(pred_phys["surface_friction"].numpy(), axis=-1)
            for name, fn in SCALARS.items():
                per_sample[name].append(float(fn(cp, wss)))

    for name in SCALARS:
        arr = np.array(per_sample[name])
        pred_means[name].append(float(arr.mean()))
        pred_stds[name].append(float(arr.std()))

    if (j + 1) % 5 == 0 or j == n_cal - 1:
        print(f"  [{j + 1}/{n_cal}]")

In [ ]:
fig, axes = plt.subplots(1, len(SCALARS), figsize=(5 * len(SCALARS), 5))

print(f"\n{'quantity':<14} {'R^2':>8} {'cov@1sig':>10} {'|z|':>8}")
print("-" * 42)

for ax, name in zip(axes, SCALARS):
    t = np.array(targets[name])
    mu = np.array(pred_means[name])
    sd = np.array(pred_stds[name])

    # R^2 (mean-prediction)
    ss_res = float(((t - mu) ** 2).sum())
    ss_tot = float(((t - t.mean()) ** 2).sum())
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else float("nan")

    # calibration
    eps = 1e-12
    within = float(np.mean(np.abs(t - mu) <= sd))
    zabs = float(np.mean(np.abs(t - mu) / (sd + eps)))

    # plot: target vs pred_mean with pred_std as error bar
    ax.errorbar(t, mu, yerr=sd, fmt="o", ms=5, alpha=0.7, capsize=3, color="tab:blue")
    lo = float(min(t.min(), (mu - sd).min()))
    hi = float(max(t.max(), (mu + sd).max()))
    ax.plot([lo, hi], [lo, hi], "k--", alpha=0.5, label="y=x")
    ax.set_xlabel(f"target {name}")
    ax.set_ylabel(f"pred {name} (mean ± std)")
    ax.set_title(f"{name}\nR²={r2:.3f}, cov@1σ={within:.2f}")
    ax.legend(loc="best", fontsize=8)
    ax.grid(alpha=0.3)

    print(f"{name:<14} {r2:>8.3f} {within:>10.2f} {zabs:>8.2f}")

plt.tight_layout()
plt.show()
print("\nideal: cov@1σ ≈ 0.68, |z| ≈ 0.80 (half-normal)")